# OptFlow on Colab GPU
Runtime → Change runtime type → **T4 GPU**. Replace the repo URL with yours.

In [4]:
!nvidia-smi
!git clone -b ashik/optflow-code https://github.com/nandakishordev7/OptFlow-Simulation.git optflow-code
%cd optflow-code
!pip install -q -r requirements.txt

Fri Sep 25 06:24:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# 1. smoke test (no data)
!python scripts/demo_synthetic.py

it    0  loss 0.08696  fit 0.08696  rigid 0.00000  thresh 2.00
it   50  loss 0.03586  fit 0.03036  rigid 0.00550  thresh 2.00
it  100  loss 0.02182  fit 0.01461  rigid 0.00722  thresh 1.00
it  150  loss 0.01747  fit 0.00975  rigid 0.00773  thresh 1.00
it  200  loss 0.01582  fit 0.00809  rigid 0.00773  thresh 0.50
it  250  loss 0.01522  fit 0.00772  rigid 0.00751  thresh 0.50
it  300  loss 0.01501  fit 0.00759  rigid 0.00742  thresh 0.25
it  350  loss 0.01151  fit 0.00585  rigid 0.00566  thresh 0.25
it  400  loss 0.01033  fit 0.00500  rigid 0.00533  thresh 0.20
it  450  loss 0.00940  fit 0.00426  rigid 0.00515  thresh 0.20
it  500  loss 0.00931  fit 0.00423  rigid 0.00508  thresh 0.20
it  550  loss 0.00935  fit 0.00427  rigid 0.00508  thresh 0.20

stopped after 557 iterations, 16.61s
zero-flow baseline: {'EPE': 1.1126, 'Acc5': 0.0, 'Acc10': 0.0, 'Outliers': 100.0, 'AngleErr': 1.5708}
OptFlow           : {'EPE': 0.104, 'Acc5': 66.2433, 'Acc10': 82.6575, 'Outliers': 18.0752, 'AngleErr': 0

In [6]:
# 2. download KITTI and check the file format
!python scripts/download_data.py kitti
!python scripts/check_data.py data/kitti

downloading kitti ...
Downloading...
From (original): https://drive.google.com/uc?id=1pjShY0RxHp0EjkelWGLO_BMR7qCB528p
From (redirected): https://drive.google.com/uc?id=1pjShY0RxHp0EjkelWGLO_BMR7qCB528p&confirm=t&uuid=e5e284f2-6bfb-4f08-812e-38b941042eb7
To: /content/optflow-code/data/kitti.download
100% 279M/279M [00:05<00:00, 48.3MB/s] 
extracting ...
done: 150 .npz files in /content/optflow-code/data/kitti
150 files
  gt           (33000, 3) float64
  pos2         (33121, 3) float64
  pos1         (33000, 3) float64


In [7]:
# 3. quick run on 5 samples, then the full benchmark
!python scripts/evaluate.py --data data/kitti --config configs/kitti.yaml --n_points 2048 --limit 5
!python scripts/evaluate.py --data data/kitti --config configs/kitti.yaml --n_points 2048 --out results_kitti_2048.json

100% 5/5 [00:24<00:00,  4.90s/it]
{
  "EPE": 0.08973865807056428,
  "Acc5": 55.439453125,
  "Acc10": 76.8359375,
  "Outliers": 54.19921875,
  "AngleErr": 0.16045098155736923,
  "time": 4.881152868270874
}
100% 150/150 [11:28<00:00,  4.59s/it]
{
  "EPE": 0.2040414520104726,
  "Acc5": 32.310221354166664,
  "Acc10": 54.195638020833336,
  "Outliers": 63.483723958333336,
  "AngleErr": 0.29218403143187366,
  "time": 4.575198322931925
}


In [8]:
# 4. ablation (Table 4)
!python scripts/ablation.py --data data/kitti --n_points 2048 --limit 50

experiment                            EPE     Acc5    Acc10    time
(a) w/o ego-motion T               0.5330    15.27    25.67   3.59s
(b) w/o adaptive thresh (2m)       0.1714    33.90    56.53   4.76s
(c) w/o correlation matrix         0.2837    10.79    28.23   4.15s
full method                        0.1615    34.64    57.29   4.66s


In [9]:
# 5. save results to Drive (Colab storage is wiped when the session ends)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/optflow_results && cp results*.json /content/drive/MyDrive/optflow_results/

Mounted at /content/drive
